# 05 Gold: `risk_forecast` (the ML step)

Which open milestones are most likely to finish late in the next 90 days? This notebook trains and evaluates a small forecast, then turns it into one row per open milestone in `mfg_mc_se_sa.bluerridge_gold_pm.risk_forecast`. The app's Risk forecast card, the daily brief and Genie all read that table.

## Method, in plain language

1. **Build a history.** The `schedule` table holds one row per milestone with static dates, not a daily delay history. So we build a time series from the milestones that are already finished (`actual_end` is set): for each **ISO week** and each **discipline** (mechanical, electrical, structural, piping, instrumentation) we take the average `slip_days` of milestones that finished that week. A week with no finished milestone has no observation, so we **forward-fill** it with the last known weekly average, starting from each discipline's first observed week. Every discipline has at least 20 non-empty weeks, so we keep one series per discipline (if one had fewer, the code falls back to a single overall series). The number of observed and filled weeks is printed below.
2. **Forecast.** We call the Databricks function **`ai_forecast`** for the next **13 weeks** (about 90 days after the as-of date, 2026-11-06), per discipline, with an 80 percent prediction interval. `ai_forecast` is a table-valued function; serverless notebook compute cannot call it directly, so the notebook sends the SQL to the SQL warehouse through the Statement Execution API (the series is first saved as a small table the warehouse can read).
3. **Check it honestly.** Backtest: hold out the last **8 weeks**, forecast them from the earlier weeks with the same method, and compare with what happened: MAE, bias (mean of forecast minus actual) and how often the actual fell inside the interval. Only weeks with a real observation count in the backtest, not forward-filled weeks. A "repeat the last value" baseline is reported next to it.
4. **Project each open milestone.** For every milestone that is still open and whose `planned_end` falls in the 90 days after the as-of date, take its discipline's forecast for the week of its planned end as its projected slip. Optional adjustment (documented, on): if any upstream dependency is still open and already late, the milestone inherits the largest such delay (days past that dependency's planned end), because it cannot finish on time if its input has not.
5. **Slip probability is a heuristic.** `slip_probability` = P(slip > 0), using a normal distribution whose mean is the projected slip and whose spread (sigma) is backed out of the 80 percent interval width (width / (2 x 1.2816)). It is derived from a trend forecast of past completions, not a calibrated per-milestone model; read it as a ranking signal, not a literal chance. `risk_rank` 1 is the highest probability; ties go to higher criticality, then earlier planned end.

Note: milestones already past their planned end and still open (for example the cryogenic tank gate MS-281, planned 2026-11-02) are outside the "planned end within the next 90 days" window, so they are not scored here; they are already late and show in the KPI strip.

In [1]:
%pip install -q mlflow
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.16.0 which is incompatible.
googleapis-common-protos 1.62.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import json, math, os
from datetime import date, timedelta, datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
dbutils.widgets.text("as_of_date", "2026-11-06", "As-of date")
dbutils.widgets.text("warehouse_id", "527214d7319cb124", "SQL warehouse for ai_forecast")
catalog = dbutils.widgets.get("catalog"); prefix = dbutils.widgets.get("schema_prefix")
AS_OF = dbutils.widgets.get("as_of_date"); WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
SILVER = f"{catalog}.{prefix}_silver_engineering"; GOLD = f"{catalog}.{prefix}_gold_pm"
HORIZON_DAYS = 90; HORIZON_WEEKS = 13; HOLDOUT_WEEKS = 8; INTERVAL = 0.8; MIN_WEEKS = 20
Z = 1.2815515655446004  # standard normal quantile for an 80 percent central interval
as_of = date.fromisoformat(AS_OF)
w = WorkspaceClient()
os.makedirs("outputs", exist_ok=True)

def wh_sql(stmt):
    r = w.statement_execution.execute_statement(statement=stmt, warehouse_id=WAREHOUSE_ID, wait_timeout="50s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(f"warehouse statement {r.status.state}: {r.status.error}")
    cols = [c.name for c in r.manifest.schema.columns]
    return pd.DataFrame(r.result.data_array or [], columns=cols)
print("as_of", AS_OF, "| warehouse", WAREHOUSE_ID)

as_of 2026-11-06 | warehouse 527214d7319cb124


## 1. Weekly slip series per discipline

In [3]:
raw = spark.sql(f'''
  SELECT discipline, date_trunc('week', actual_end) AS ds, avg(slip_days) AS y, count(*) AS n_done
  FROM {SILVER}.schedule
  WHERE actual_end IS NOT NULL AND actual_end <= DATE'{AS_OF}'
  GROUP BY 1, 2''').toPandas()
raw["ds"] = pd.to_datetime(raw["ds"]); raw["y"] = raw["y"].astype(float)
last_week = pd.Timestamp(as_of) - pd.Timedelta(days=as_of.weekday())   # Monday of the as-of week
parts = []
for disc, g in raw.groupby("discipline"):
    grid = pd.date_range(g["ds"].min(), last_week, freq="7D")
    s = g.set_index("ds").reindex(grid)
    s["observed"] = s["y"].notna()
    s["y"] = s["y"].ffill()
    s["discipline"] = disc
    parts.append(s.rename_axis("ds").reset_index()[["discipline", "ds", "y", "observed"]])
series = pd.concat(parts, ignore_index=True)
summary = series.groupby("discipline").agg(weeks=("ds", "size"), observed_weeks=("observed", "sum"), first_week=("ds", "min"), last_week=("ds", "max"))
print(summary.to_string())
GROUPED = bool((summary["observed_weeks"] >= MIN_WEEKS).all())
if not GROUPED:   # fallback: one overall series
    o = raw.assign(w_=raw["y"] * raw["n_done"]).groupby("ds").agg(w_=("w_", "sum"), n=("n_done", "sum"))
    o["y"] = o["w_"] / o["n"]
    grid = pd.date_range(o.index.min(), last_week, freq="7D"); o = o.reindex(grid)
    series = pd.DataFrame({"discipline": "all", "ds": grid, "y": o["y"].ffill().values, "observed": o["y"].notna().values})
print("grouped by discipline:", GROUPED, "| total series rows:", len(series))
SERIES_LEN = int(series.groupby("discipline").size().median())

            weeks  observed_weeks first_week  last_week
discipline                                             
electrical     55              33 2025-10-20 2026-11-02
instr          55              33 2025-10-20 2026-11-02
mechanical     54              36 2025-10-27 2026-11-02
piping         54              25 2025-10-27 2026-11-02
structural     55              30 2025-10-20 2026-11-02
grouped by discipline: True | total series rows: 273


## 2. `ai_forecast` through the SQL warehouse, and the backtest

`ai_forecast` needs a SQL warehouse, so each series is saved as a table and the query is run through the Statement Execution API. The backtest uses the identical call on the series minus its last 8 weeks.

In [4]:
def save_series(df, name):
    sdf = spark.createDataFrame(df[["discipline", "ds", "y"]].assign(ds=df["ds"].dt.date, y=df["y"].astype(float)))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{name}")

def run_ai_forecast(table, horizon_date):
    sql = (f"SELECT discipline, ds, y_forecast, y_lower, y_upper FROM ai_forecast(TABLE(SELECT discipline, ds, y FROM {GOLD}.{table}), "
           f"horizon => DATE'{horizon_date}', time_col => 'ds', value_col => 'y', group_col => 'discipline', "
           f"prediction_interval_width => {INTERVAL})")
    print("warehouse SQL:", sql)
    f = wh_sql(sql)
    f["ds"] = pd.to_datetime(f["ds"]).dt.tz_localize(None).dt.normalize()
    for c in ["y_forecast", "y_lower", "y_upper"]: f[c] = f[c].astype(float)
    return f

all_weeks = sorted(series["ds"].unique())
cut = all_weeks[-HOLDOUT_WEEKS]                       # first held-out week
train, hold = series[series["ds"] < cut], series[series["ds"] >= cut]
save_series(train, "risk_series_train"); save_series(series, "risk_series")

bt = run_ai_forecast("risk_series_train", pd.Timestamp(all_weeks[-1]).date())
bt = bt.merge(hold, on=["discipline", "ds"], how="inner")
obs = bt[bt["observed"]].copy()                       # score only weeks with a real observation
obs["err"] = obs["y_forecast"] - obs["y"]
obs["inside"] = (obs["y"] >= obs["y_lower"]) & (obs["y"] <= obs["y_upper"])
# baseline: repeat the last training value of each discipline
last_train = train.sort_values("ds").groupby("discipline")["y"].last()
obs["naive_err"] = obs["discipline"].map(last_train) - obs["y"]
mae, bias, cov = float(obs["err"].abs().mean()), float(obs["err"].mean()), float(obs["inside"].mean())
naive_mae, naive_bias = float(obs["naive_err"].abs().mean()), float(obs["naive_err"].mean())
print(f"holdout weeks={HOLDOUT_WEEKS} (from {pd.Timestamp(cut).date()}), scored observed points={len(obs)} of {len(bt)}")
print(f"BACKTEST  MAE={mae:.3f} d   bias={bias:+.3f} d   coverage@{int(INTERVAL*100)}%={cov:.2%}")
print(f"BASELINE  last-value MAE={naive_mae:.3f} d   bias={naive_bias:+.3f} d")
per_disc = obs.groupby("discipline").agg(n=("err", "size"), mae=("err", lambda e: e.abs().mean()), bias=("err", "mean"), coverage=("inside", "mean")).round(3)
print(per_disc.to_string())

warehouse SQL: SELECT discipline, ds, y_forecast, y_lower, y_upper FROM ai_forecast(TABLE(SELECT discipline, ds, y FROM mfg_mc_se_sa.bluerridge_gold_pm.risk_series_train), horizon => DATE'2026-11-02', time_col => 'ds', value_col => 'y', group_col => 'discipline', prediction_interval_width => 0.8)
holdout weeks=8 (from 2026-09-14), scored observed points=12 of 35
BACKTEST  MAE=11.048 d   bias=-10.091 d   coverage@80%=16.67%
BASELINE  last-value MAE=11.347 d   bias=-9.903 d
            n     mae    bias  coverage
discipline                             
electrical  1   3.693   3.693     0.000
instr       4   6.826  -6.826     0.250
mechanical  6  13.874 -13.190     0.167
piping      1  18.340 -18.340     0.000


## 3. Full forecast, 13 weeks past the as-of date

In [5]:
# 90 days after 2026-11-06 is 2027-02-04, inside the week of 2027-02-01; ask for the horizon date one week beyond so that week is returned,
# then keep only weeks starting on or before as_of + 90 days (13 forecast weeks).
horizon_date = (pd.Timestamp(last_week) + pd.Timedelta(weeks=HORIZON_WEEKS + 1)).date()
fc = run_ai_forecast("risk_series", horizon_date)
fc = fc[fc["ds"] <= pd.Timestamp(as_of) + pd.Timedelta(days=HORIZON_DAYS)].reset_index(drop=True)
print("forecast rows:", len(fc), "| weeks per discipline:", fc.groupby("discipline").size().to_dict(), "| last forecast week:", fc["ds"].max().date())
print(fc.groupby("discipline")[["y_forecast", "y_lower", "y_upper"]].mean().round(2).to_string())

warehouse SQL: SELECT discipline, ds, y_forecast, y_lower, y_upper FROM ai_forecast(TABLE(SELECT discipline, ds, y FROM mfg_mc_se_sa.bluerridge_gold_pm.risk_series), horizon => DATE'2027-02-08', time_col => 'ds', value_col => 'y', group_col => 'discipline', prediction_interval_width => 0.8)
forecast rows: 65 | weeks per discipline: {'electrical': 13, 'instr': 13, 'mechanical': 13, 'piping': 13, 'structural': 13} | last forecast week: 2027-02-01
            y_forecast  y_lower  y_upper
discipline                              
electrical        2.35    -1.33     8.50
instr             7.20    -0.37    13.34
mechanical       14.84    -0.08    29.48
piping            1.29    -1.31    14.66
structural       -0.66    -2.62     1.41


## 4. Project every open milestone

Open = no `actual_end`. In scope = `planned_end` in (as-of, as-of + 90 days]. Each gets its discipline's forecast for the week of its planned end, plus the optional upstream-delay adjustment described above.

In [6]:
ms = spark.sql(f'''
  SELECT milestone_id, name, discipline, criticality, planned_end, depends_on_ids
  FROM {SILVER}.schedule
  WHERE actual_end IS NULL AND planned_end > DATE'{AS_OF}' AND planned_end <= date_add(DATE'{AS_OF}', {HORIZON_DAYS})''').toPandas()
# days late so far for every still-open, overdue milestone (candidate late upstream dependencies)
late = spark.sql(f'''
  SELECT milestone_id, datediff(DATE'{AS_OF}', planned_end) AS days_late
  FROM {SILVER}.schedule WHERE actual_end IS NULL AND planned_end < DATE'{AS_OF}' ''').toPandas().set_index("milestone_id")["days_late"].to_dict()
def upstream_delay(deps):
    return float(max([late.get(d, 0) for d in (list(deps) if deps is not None else [])] + [0]))
ms["upstream_delay"] = ms["depends_on_ids"].apply(upstream_delay)
ms["ds"] = pd.to_datetime(ms["planned_end"]).apply(lambda d: d - pd.Timedelta(days=d.weekday())).clip(lower=fc["ds"].min())   # 7-8 Nov falls in the as-of week: use the first forecast week
ms["_g"] = ms["discipline"] if GROUPED else "all"
m = ms.merge(fc.rename(columns={"discipline": "_g"}), on=["_g", "ds"], how="left")
assert m["y_forecast"].notna().all(), "a milestone week has no forecast: " + str(m[m["y_forecast"].isna()][["milestone_id","_g","ds"]].head().to_dict("records")) + " fc dtypes " + str(fc.dtypes.to_dict()) + str(fc.head(2).to_dict("records")) + str(ms.dtypes.to_dict())
m["projected_slip_days"] = m["y_forecast"] + m["upstream_delay"]
m["forecast_confidence_lower"] = m["y_lower"] + m["upstream_delay"]
m["forecast_confidence_upper"] = m["y_upper"] + m["upstream_delay"]
sigma = ((m["y_upper"] - m["y_lower"]) / (2 * Z)).clip(lower=1e-6)
m["slip_probability"] = [min(1.0, max(0.0, 0.5 * (1 + math.erf(mu / (s * math.sqrt(2)))))) for mu, s in zip(m["projected_slip_days"], sigma)]
m = m.sort_values(["slip_probability", "criticality", "planned_end"], ascending=[False, False, True]).reset_index(drop=True)
m["risk_rank"] = np.arange(1, len(m) + 1)    # strict order after the tie-break
METHOD = "ai_forecast weekly slip trend per discipline (13 wk, 80% interval) + normal approx P(slip>0) + upstream-late adjustment"
out = m.assign(forecast_date=pd.Timestamp(as_of).date(), horizon_days=HORIZON_DAYS, method=METHOD,
               computed_at=datetime.now(timezone.utc).replace(tzinfo=None),
               planned_end=pd.to_datetime(m["planned_end"]).dt.date)[
    ["milestone_id", "name", "discipline", "criticality", "planned_end", "forecast_date", "horizon_days", "projected_slip_days",
     "forecast_confidence_lower", "forecast_confidence_upper", "slip_probability", "risk_rank", "method", "computed_at"]]
print("open milestones scored:", len(out), "| with an upstream-late adjustment:", int((m["upstream_delay"] > 0).sum()))

open milestones scored: 64 | with an upstream-late adjustment: 6


## 5. Write `risk_forecast` (primary key `milestone_id`)

In [7]:
spark.sql(f"DROP TABLE IF EXISTS {GOLD}.risk_forecast")
spark.sql(f'''
CREATE TABLE {GOLD}.risk_forecast (
  milestone_id STRING NOT NULL COMMENT 'Open milestone, primary key',
  name STRING, discipline STRING, criticality INT,
  planned_end DATE COMMENT 'Planned finish, inside (as_of, as_of + 90 days]',
  forecast_date DATE COMMENT 'The as-of date the forecast was made',
  horizon_days INT COMMENT 'Forecast horizon in days (90)',
  projected_slip_days DOUBLE COMMENT 'Discipline trend forecast for the planned-end week plus upstream-late delay',
  forecast_confidence_lower DOUBLE COMMENT 'Lower end of the 80 percent interval',
  forecast_confidence_upper DOUBLE COMMENT 'Upper end of the 80 percent interval',
  slip_probability DOUBLE COMMENT 'Heuristic P(slip > 0), normal approximation from the forecast interval; a ranking signal',
  risk_rank INT COMMENT '1 = highest slip_probability; ties: criticality desc, planned_end asc',
  method STRING, computed_at TIMESTAMP,
  CONSTRAINT risk_forecast_pk PRIMARY KEY (milestone_id))
COMMENT 'Schedule-slip risk forecast for open milestones due in the next 90 days (ai_forecast on weekly slip trend). Heuristic probability.'
''')
spark.createDataFrame(out).createOrReplaceTempView("_risk_out")
spark.sql(f"INSERT INTO {GOLD}.risk_forecast SELECT milestone_id, name, discipline, CAST(criticality AS INT), planned_end, forecast_date, CAST(horizon_days AS INT), "
          "projected_slip_days, forecast_confidence_lower, forecast_confidence_upper, slip_probability, CAST(risk_rank AS INT), method, CAST(computed_at AS TIMESTAMP) FROM _risk_out")
for t in ("risk_series", "risk_series_train"):
    spark.sql(f"COMMENT ON TABLE {GOLD}.{t} IS 'Helper: weekly mean slip per discipline (forward-filled) read by ai_forecast in 02_risk_forecast'")
print("rows written:", spark.table(f"{GOLD}.risk_forecast").count())

rows written: 64


## 6. Asserts

In [8]:
rf = spark.table(f"{GOLD}.risk_forecast").toPandas()
sched = spark.table(f"{SILVER}.schedule").toPandas().set_index("milestone_id")
assert len(rf) > 0
assert sched.loc[rf["milestone_id"], "actual_end"].isna().all(), "a scored milestone is not open"
assert rf["slip_probability"].between(0, 1).all()
assert ((rf["forecast_confidence_lower"] <= rf["projected_slip_days"]) & (rf["projected_slip_days"] <= rf["forecast_confidence_upper"])).all()
assert (rf["horizon_days"] == 90).all()
pe = pd.to_datetime(rf["planned_end"])
assert (pe > pd.Timestamp(as_of)).all() and (pe <= pd.Timestamp(as_of) + pd.Timedelta(days=HORIZON_DAYS)).all()
assert rf["milestone_id"].is_unique and rf["milestone_id"].notna().all()
assert sorted(rf["risk_rank"]) == list(range(1, len(rf) + 1)), "risk_rank must be dense from 1"
assert math.isfinite(mae) and math.isfinite(bias) and math.isfinite(cov)
assert (rf.sort_values("risk_rank")["slip_probability"].diff().dropna() <= 1e-12).all(), "rank must follow probability"
print("all asserts passed:", len(rf), "open milestones; probability range", round(rf.slip_probability.min(), 3), "to", round(rf.slip_probability.max(), 3))

all asserts passed: 64 open milestones; probability range 0.313 to 1.0


## 7. Top 5 risks

In [9]:
top5 = spark.sql(f'''SELECT risk_rank, milestone_id, name, discipline, criticality, planned_end,
  round(projected_slip_days,1) AS projected_slip_days, round(forecast_confidence_lower,1) AS lower, round(forecast_confidence_upper,1) AS upper,
  round(slip_probability,3) AS slip_probability FROM {GOLD}.risk_forecast ORDER BY risk_rank LIMIT 5''')
display(top5)
t = top5.toPandas()
dep = sched.loc[t["milestone_id"], "depends_on_ids"]
print("upstream-late adjustment on top 5:", {mid: upstream_delay(d) for mid, d in dep.items()})
print("MS-281 and its dependents (any open milestone listing MS-281 upstream):",
      [i for i, d in sched["depends_on_ids"].items() if d is not None and "MS-281" in list(d)])

risk_rank,milestone_id,name,discipline,criticality,planned_end,projected_slip_days,lower,upper,slip_probability
1,MS-299,Terminate cables - Tank T-203 (WP-297),electrical,3,2026-12-04,17.0,13.5,22.9,1.0
2,MS-297,Hydrotest line - Flare and Vent (WP-269),piping,3,2026-11-19,16.2,13.6,28.1,0.998
3,MS-303,Calibrate transmitters - Train 4 (WP-310),instr,3,2027-01-03,12.4,4.8,18.6,0.989
4,MS-335,Loop check - Train 4 (WP-315),instr,1,2027-01-25,8.3,0.7,14.8,0.935
5,MS-351,Loop check - Tank T-203 (WP-364),instr,3,2027-01-21,8.1,0.5,14.5,0.93


upstream-late adjustment on top 5: {'MS-299': 15.0, 'MS-297': 15.0, 'MS-303': 5.0, 'MS-335': 0.0, 'MS-351': 0.0}
MS-281 and its dependents (any open milestone listing MS-281 upstream): ['MS-304', 'MS-340']


## 8. Chart: history, forecast with interval band, and the backtest overlay

In [10]:
discs = sorted(series["discipline"].unique())
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for ax, d in zip(axes.ravel(), discs):
    h = series[series["discipline"] == d]
    ax.plot(h["ds"], h["y"], color="#0B2026", lw=1.4, label="weekly mean slip (history)")
    ho = h[h["observed"] & (h["ds"] >= cut)]
    ax.scatter(ho["ds"], ho["y"], color="#0B2026", s=14, zorder=3)
    b = bt[bt["discipline"] == d]
    ax.fill_between(b["ds"], b["y_lower"], b["y_upper"], color="#6C8EAD", alpha=0.30, label="backtest 80% band")
    ax.plot(b["ds"], b["y_forecast"], color="#2F6F9F", lw=1.4, ls="--", label="backtest forecast")
    f = fc[fc["discipline"] == d]
    ax.fill_between(f["ds"], f["y_lower"], f["y_upper"], color="#FF3621", alpha=0.18, label="forecast 80% band")
    ax.plot(f["ds"], f["y_forecast"], color="#FF3621", lw=1.6, label="forecast (13 wk)")
    ax.axvline(pd.Timestamp(as_of), color="#888", lw=0.8, ls=":")
    ax.set_title(d, loc="left", fontsize=11); ax.set_ylabel("slip (days)"); ax.grid(alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.ravel()[len(discs):]: ax.axis("off")
h_, l_ = axes.ravel()[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower right", bbox_to_anchor=(0.97, 0.12), frameon=False)
fig.suptitle(f"Blue Ridge LNG: weekly schedule slip by discipline, backtest and 13-week ai_forecast (as of {AS_OF})", x=0.01, ha="left", fontsize=12)
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig("outputs/risk_forecast_chart.png", dpi=130); plt.close(fig)
print("saved outputs/risk_forecast_chart.png", os.path.getsize("outputs/risk_forecast_chart.png"), "bytes")

saved outputs/risk_forecast_chart.png 214085 bytes


![Weekly slip by discipline: history, backtest and 13-week ai_forecast](outputs/risk_forecast_chart.png)

The chart is also committed at `05_gold/outputs/risk_forecast_chart.png`. Dark line: weekly mean slip (history); dashed blue line and band: the 8-week backtest forecast and its 80 percent interval; red line and band: the 13-week forecast from the as-of date (dotted vertical line).

## 9. MLflow and the metrics file

In [11]:
import mlflow
exp_path = "/Shared/bluerridge_pm_copilot/schedule_risk_forecast"
w.workspace.mkdirs("/Shared/bluerridge_pm_copilot")
mlflow.set_tracking_uri("databricks"); mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(exp_path)
try:
    job_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().toString()
except Exception:
    job_run_id = "unknown"
params = {"method": "ai_forecast", "grain": "weekly (ISO week, Monday start)", "grouped_by": "discipline" if GROUPED else "overall",
          "series_length_weeks": SERIES_LEN, "holdout_weeks": HOLDOUT_WEEKS, "horizon_days": HORIZON_DAYS, "horizon_weeks": HORIZON_WEEKS,
          "prediction_interval_width": INTERVAL, "as_of_date": AS_OF, "invocation": "warehouse via Statement Execution API"}
with mlflow.start_run(run_name=f"risk_forecast_{AS_OF}") as run:
    mlflow.log_params(params)
    mlflow.log_metrics({"backtest_mae_days": mae, "backtest_bias_days": bias, "backtest_interval_coverage": cov,
                        "baseline_last_value_mae_days": naive_mae, "milestones_scored": float(len(rf))})
    mlflow.log_artifact("outputs/risk_forecast_chart.png")
    mlflow_run_id = run.info.run_id
metrics = {
    "method": "ai_forecast", "method_detail": METHOD, "ai_forecast_path": "SQL warehouse via Statement Execution API (table-valued function; series saved as tables first)",
    "series_grain": "weekly (ISO week) mean slip_days of milestones finished that week, forward-filled for empty weeks",
    "grouped_by": "discipline" if GROUPED else "overall", "series_length_weeks_median_per_group": SERIES_LEN,
    "observed_weeks_per_discipline": {k: int(v) for k, v in summary["observed_weeks"].items()},
    "holdout_weeks": HOLDOUT_WEEKS, "holdout_start": str(pd.Timestamp(cut).date()), "holdout_points_scored": int(len(obs)),
    "mae_days": round(mae, 4), "bias_days": round(bias, 4), "interval_coverage": round(cov, 4),
    "baseline_last_value_mae_days": round(naive_mae, 4), "baseline_last_value_bias_days": round(naive_bias, 4),
    "per_discipline": json.loads(per_disc.reset_index().to_json(orient="records")),
    "horizon_days": HORIZON_DAYS, "horizon_weeks": HORIZON_WEEKS, "as_of_date": AS_OF, "prediction_interval_width": INTERVAL,
    "milestones_scored": int(len(rf)), "slip_probability_note": "heuristic normal approximation from the forecast interval, not calibrated",
    "job_run_id": job_run_id, "mlflow_experiment_path": exp_path, "mlflow_experiment_id": exp.experiment_id, "mlflow_run_id": mlflow_run_id,
}
with open("outputs/risk_model_metrics.json", "w") as fh: json.dump(metrics, fh, indent=2)
print(json.dumps(metrics, indent=2))

2026/11/08 16:44:49 INFO mlflow.tracking.fluent: Experiment with name '/Shared/bluerridge_pm_copilot/schedule_risk_forecast' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


{
  "method": "ai_forecast",
  "method_detail": "ai_forecast weekly slip trend per discipline (13 wk, 80% interval) + normal approx P(slip>0) + upstream-late adjustment",
  "ai_forecast_path": "SQL warehouse via Statement Execution API (table-valued function; series saved as tables first)",
  "series_grain": "weekly (ISO week) mean slip_days of milestones finished that week, forward-filled for empty weeks",
  "grouped_by": "discipline",
  "series_length_weeks_median_per_group": 55,
  "observed_weeks_per_discipline": {
    "electrical": 33,
    "instr": 33,
    "mechanical": 36,
    "piping": 25,
    "structural": 30
  },
  "holdout_weeks": 8,
  "holdout_start": "2026-09-14",
  "holdout_points_scored": 12,
  "mae_days": 11.0483,
  "bias_days": -10.0911,
  "interval_coverage": 0.1667,
  "baseline_last_value_mae_days": 11.3472,
  "baseline_last_value_bias_days": -9.9028,
  "per_discipline": [
    {
      "discipline": "electrical",
      "n": 1,
      "mae": 3.693,
      "bias": 3.693,
  

## How to read the result (honest read)

- **The backtest is weak, and that is the finding.** MAE is about 11 days and bias about -10 days (the forecast is too low), and only about 17 percent of observed held-out weeks fall inside the 80 percent band (target 80). The "repeat the last value" baseline scores about the same MAE (11.3), so `ai_forecast` adds little over the naive guess here. Cause: the last eight weeks contain far fewer completions than the earlier history (12 observed points across four disciplines; structural had none), and the weekly means in that stretch jumped above the earlier trend, which any trend model trained on the earlier weeks misses.
- Treat the forecast as a **ranking signal** for the PM, not a calibrated chance. Because the backtest under-forecasts, the published slip days probably err low rather than high.
- **Top-risk plausibility.** Ranks 1 to 5 are all driven by discipline trend plus upstream milestones that are still open and late (MS-271, 15 days late; MS-277, 11 days late), and two of the five are Tank T-203 work packages (MS-299, MS-351). The cryogenic tank gate MS-281 itself is 4 days late and past its planned end, so it falls outside the "planned end in the next 90 days" window and is not scored. Its two downstream milestones that are scored, MS-304 (Erect piping - Jetty 2) and MS-340 (Cryogenic Piping Welding and NDE Complete, a key milestone), rank 29 and 30 of 64 with slip probability about 0.8, so they are flagged but not near the top. Nothing was tuned to move any milestone.
- `job_run_id` in the metrics JSON reads "unknown" because the notebook could not read its own run id on serverless; the one-time submit run id is recorded in this notebook's metadata (`evidence`) and in the task report.